# Notebook ٣: scikit-learn للنصوص (Intent Classifier)

بنبني الجزء اللي بيفهم نية المستخدم في شات بوت كورة. كل خلية اتشغلت فعلًا، scikit-learn 1.7.

## ١. الداتا: جمل ونوايا

كل جملة عليها label بتقول المستخدم عايز إيه (intent).

In [ ]:
# مكتبة الجداول
import pandas as pd
# قاموس: النية ← جمل
data = {
# جمل عن الإحصائيات
 "stats": ["how many goals did salah score", "show me haaland stats", "assists for rice this season", "top scorer in the league", "how many minutes did saka play", "give me player points", "who has the most assists", "goals and assists of palmer", "what are the stats for son", "clean sheets for raya", "points per game of watkins", "best player this season"],
# جمل عن الإصابات
 "injury": ["is salah injured", "any injury news for saka", "is haaland fit to play", "who is out with injury", "when will rice return", "ankle injury update", "is he suspended", "knock on the hamstring", "fitness news for the arsenal squad", "is palmer doubtful", "who is ruled out this week", "injured players list"],
# جمل عن الانتقالات
 "transfer": ["did he join a new club", "any transfer news", "who left on loan", "is salah leaving liverpool", "new signing at arsenal", "loan deal for the winger", "who joined chelsea", "player sold this summer", "transfer window rumours", "has he signed a contract", "who moved on loan", "latest signings"],
# جمل عن المباريات
 "fixtures": ["when is the next match", "who do arsenal play next", "fixtures for this week", "what time is the game", "next gameweek schedule", "when does liverpool play", "show upcoming fixtures", "is there a match tomorrow", "arsenal vs chelsea date", "kick off time today", "next opponent for city", "match calendar"],
# تحيات
 "greeting": ["hello", "hi there", "good morning", "hey bot", "hello how are you", "hi", "good evening", "hey there", "hello again", "morning", "hi friend", "hey"],
# نقفل القاموس
}
# نحوله لقايمة (جملة، نية)
rows = [(t, k) for k, v in data.items() for t in v]
# نعمل DataFrame
df = pd.DataFrame(rows, columns=["text", "intent"])
# الشكل
print(df.shape)
# عدد الجمل لكل نية
df["intent"].value_counts()


## ٢. تحويل النص لأرقام: Bag of Words

الموديل مبيفهمش كلام. بنعد الكلمات في كل جملة.

In [ ]:
# معدّ الكلمات
from sklearn.feature_extraction.text import CountVectorizer
# 3 جمل صغيرة للتجربة
small = ["salah goals", "salah injured", "next match"]
# ننشئ الـ vectorizer
cv = CountVectorizer()
# بيتعلم المفردات ويحوّل الجمل
X = cv.fit_transform(small)
# المفردات اللي اتعلمها
print(cv.get_feature_names_out())
# مصفوفة العد: صف لكل جملة
print(X.toarray())


## ٣. TF-IDF

بيقلل وزن الكلمات الشائعة ويرفع الكلمات المميزة.

In [ ]:
# معدّ TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
# ننشئه
tf = TfidfVectorizer()
# نحول كل جمل الداتا
Xt = tf.fit_transform(df["text"])
# (عدد الجمل، عدد المفردات)
print(Xt.shape)
# نسبة الخانات غير الصفرية (مصفوفة sparse)
print(round(Xt.nnz / (Xt.shape[0] * Xt.shape[1]) * 100, 1), "% non-zero")


## ٤. تدريب وتقييم

نقسم الداتا لتدريب واختبار، ونتأكد إن الاختبار مش مستخدم في التدريب.

In [ ]:
# دالة تقسيم الداتا
from sklearn.model_selection import train_test_split
# بتربط خطوات المعالجة والموديل
from sklearn.pipeline import make_pipeline
# الموديل
from sklearn.linear_model import LogisticRegression
# تقرير الدقة
from sklearn.metrics import classification_report

# 75% تدريب و25% اختبار بنفس نسب النوايا
Xtr, Xte, ytr, yte = train_test_split(df["text"], df["intent"], test_size=0.25, random_state=0, stratify=df["intent"])
# TF-IDF بكلمات مفردة وثنائية ثم logistic regression (C=10 يعني تنظيم أخف)
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(C=10, max_iter=1000))
# ندرب على جمل التدريب
model.fit(Xtr, ytr)
# نتوقع نية جمل الاختبار
pred = model.predict(Xte)
# precision وrecall وf1 لكل نية
print(classification_report(yte, pred, zero_division=0))


## ٥. استخدامه كبوت: نية + ثقة

لو الثقة قليلة البوت يقول 'مش فاهم' بدل ما يغلط.

In [ ]:
# دالة بترجع النية لو الثقة كفاية
def understand(text, threshold=0.3):
# احتمال كل نية
    probs = model.predict_proba([text])[0]
# رقم أعلى احتمال
    i = probs.argmax()
# لو أقل من الحد
    if probs[i] < threshold:
# نقول مش فاهم
        return "unknown", round(float(probs[i]), 2)
# نرجع النية والثقة
    return model.classes_[i], round(float(probs[i]), 2)

# 4 أسئلة تجريبية
for q in ["is saka injured", "when do arsenal play", "hello", "what is the capital of france"]:
# نطبع السؤال والنتيجة
    print(q, "->", understand(q))


## ٦. Cross-validation

تقسيمة واحدة ممكن تضلل، فنكررها 5 مرات.

In [ ]:
# حساب الدقة بالتقسيم المتكرر
from sklearn.model_selection import cross_val_score
# 5 تقسيمات
scores = cross_val_score(model, df["text"], df["intent"], cv=5)
# دقة كل تقسيمة
print(scores.round(2))
# المتوسط
print("mean:", scores.mean().round(3))


## ٧. نحفظ الموديل

علشان نستخدمه في Streamlit من غير ما ندرب من الأول.

In [ ]:
# مكتبة حفظ الكائنات
import joblib
# نحفظ الـ pipeline كله
joblib.dump(model, "intent_model.joblib")
# نحمّله تاني
loaded = joblib.load("intent_model.joblib")
# نجرب الموديل المحمّل
print(loaded.predict(["any injury news for rice"])[0])
